In [1]:
import requests

r = requests.get('http://localhost:11434/api/tags')
models = [m['name'] for m in r.json()['models']]
print('Available models:', models)

for required in ['llava:7b', 'qwen2.5vl:7b']:
    status = '✓' if required in models else '✗ MISSING'
    print(f'  {required}: {status}')

Available models: ['qwen2.5vl:7b', 'llava:7b', 'llama3:latest']
  llava:7b: ✓
  qwen2.5vl:7b: ✓


In [2]:
import requests

def quick_test(model, prompt):
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=300  # 5 minutes instead of 60 seconds
    )
    return r.json().get("response", r.json())

In [3]:
print("Testing qwen2.5vl:7b...")
print(quick_test("qwen2.5vl:7b", "Reply with exactly one word: hello"))

Testing qwen2.5vl:7b...
Hello


In [4]:
print("Testing llava:7b...")
print(quick_test("llava:7b", "Reply with exactly one word: hello"))

Testing llava:7b...
 Hello! 


In [7]:
!python Dentist/demo.py


--- Qwen2VL | nbars ---
Q:        How many bars are there in the specified figure panel?
GT:       50
Baseline: The figure shows a histogram with 20 bars.
Revised:  20

--- LLaVA | nbars ---
Q:        How many bars are there in the specified figure panel?
GT:       50
Baseline:  The image shows a line graph with a histogram on top. However, it is not possible to determine the number of bars present on the graph without additional context or a clearer view of the data represented. If you can provide more details or a higher resolution image, I could attempt to give you a more accurate answer. 
Revised:   10 

--- Qwen2VL | minimum ---
Q:        What is the minimum value of the data in this figure panel? 
GT:       -0.37754538752755923
Baseline: The minimum value of the data in the figure panel can be determined by looking at the leftmost bar of the histogram. The x-axis is labeled with values ranging from approximately -0.35 to 0.20. The leftmost bar of the histogram starts at the valu

In [2]:
!python Dentist/Evaluation.py

Total results: 960
Models: {'Qwen2VL', 'LLaVA'}
Question types: {'nbars', 'ngaussians', 'mean', 'median', 'minimum', 'maximum'}
Sample result keys: ['model', 'image_path', 'question', 'ground_truth', 'type', 'baseline_answer', 'revised_answer']
Scoring functions defined.
Scored 960 results
     model     type  baseline_score  revised_score dentist_effect
0  Qwen2VL    nbars             0.0            0.0      unchanged
1    LLaVA    nbars             0.0            0.0      unchanged
2  Qwen2VL  minimum             1.0            1.0      unchanged
3    LLaVA  minimum             0.0            0.0      unchanged
4  Qwen2VL  maximum             0.0            0.0      unchanged
5    LLaVA  maximum             0.0            0.0      unchanged
6  Qwen2VL   median             0.0            0.0      unchanged
7    LLaVA   median             0.0            0.0      unchanged
8  Qwen2VL     mean             0.0            0.0      unchanged
9    LLaVA     mean             0.0            0.

In [3]:
import json
import pandas as pd
import numpy as np
import re

# ── load JSON ─────────────────────────────────────────────────────────────────

with open('results.json', 'r') as f:  # adjust filename if needed
    data = json.load(f)

print(f"Loaded {len(data)} items")
print(f"Keys in first item: {list(data[0].keys())}")
print(f"\nSample item:")
for k, v in data[0].items():
    print(f"  {k}: {v}")

Loaded 960 items
Keys in first item: ['model', 'image_path', 'question', 'ground_truth', 'type', 'baseline_answer', 'revised_answer']

Sample item:
  model: Qwen2VL
  image_path: /home/kbarbee2/.cache/huggingface/hub/datasets--ReadingTimeMachine--visual_qa_histograms/snapshots/f7ba35f3751cf2b27c943ccb1d313424897c913b/example_hists/imgs/imgs/id_0000.jpeg
  question: How many bars are there in the specified figure panel?
  ground_truth: 50
  type: nbars
  baseline_answer: The figure shows a histogram with 20 bars.
  revised_answer: 20


In [6]:
import json
import re
import numpy as np
import pandas as pd

with open('results.json') as f:
    results = json.load(f)

# ── helpers (identical to your original) ─────────────────────────────────────

def extract_number(text):
    if text in [None, 'ERROR', '']:
        return None
    matches = re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', str(text))
    if matches:
        return float(matches[0])
    return None


def score_numerical(predicted, ground_truth, tolerance=0.20):
    pred = extract_number(predicted)
    gt   = extract_number(str(ground_truth))
    if pred is None or gt is None:
        return 0.0
    if gt == 0:
        return 1.0 if abs(pred) < 1e-6 else 0.0
    relative_error = abs(pred - gt) / abs(gt)
    return 1.0 if relative_error <= tolerance else 0.0


def score_exact(predicted, ground_truth):
    """Exact match — same as your original."""
    pred = extract_number(predicted)
    gt   = extract_number(str(ground_truth))
    if pred is None or gt is None:
        return 0.0
    return 1.0 if int(round(pred)) == int(round(gt)) else 0.0


def score_answer(predicted, ground_truth, qtype, tolerance=0.20):
    """
    nbars and ngaussians use exact match (same as original).
    All numerical types use relative error at the given tolerance.
    """
    if qtype in ['nbars', 'ngaussians']:
        return score_exact(predicted, ground_truth)   # tolerance ignored for these
    else:
        return score_numerical(predicted, ground_truth, tolerance=tolerance)


# ── score at multiple tolerances ──────────────────────────────────────────────

tolerances = [0.10, 0.20, 0.30, 0.50]

for r in results:
    for tol in tolerances:
        tag = int(tol * 100)
        r[f'baseline_tol{tag}'] = score_answer(
            r['baseline_answer'], r['ground_truth'], r['type'], tol
        )
        r[f'revised_tol{tag}'] = score_answer(
            r['revised_answer'], r['ground_truth'], r['type'], tol
        )

    # keep original tol20 as the canonical score to match your original script
    r['baseline_score'] = r['baseline_tol20']
    r['revised_score']  = r['revised_tol20']

    if r['revised_score'] > r['baseline_score']:
        r['dentist_effect'] = 'improved'
    elif r['revised_score'] < r['baseline_score']:
        r['dentist_effect'] = 'worsened'
    else:
        r['dentist_effect'] = 'unchanged'

df = pd.DataFrame(results)
print(f'Scored {len(df)} results')

# ── verify tol20 matches your original output ─────────────────────────────────

print('\nVERIFICATION — tol20 should match your original script exactly:')
for model in ['Qwen2VL', 'LLaVA']:
    mdf = df[df['model'] == model]
    print(f'\n--- {model} ---')
    for qtype in sorted(df['type'].unique()):
        qdf = mdf[mdf['type'] == qtype]
        b = qdf['baseline_tol20'].mean()
        r = qdf['revised_tol20'].mean()
        print(f'  {qtype:<15} baseline={b:.3f}  revised={r:.3f}')

# ── full tolerance summary ────────────────────────────────────────────────────

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.width', 140)

print('\n' + '='*80)
print('ACCURACY BY MODEL, QUESTION TYPE, AND TOLERANCE')
print('='*80)

for model in ['Qwen2VL', 'LLaVA']:
    print(f'\n--- {model} ---')
    mdf = df[df['model'] == model]
    rows = []
    for qtype in sorted(mdf['type'].unique()):
        qdf  = mdf[mdf['type'] == qtype]
        row  = {'Type': qtype, 'N': len(qdf)}
        for tol in tolerances:
            tag = int(tol * 100)
            row[f'Base{tag}'] = qdf[f'baseline_tol{tag}'].mean()
            row[f'Rev{tag}']  = qdf[f'revised_tol{tag}'].mean()
            row[f'Δ{tag}']    = row[f'Rev{tag}'] - row[f'Base{tag}']
        rows.append(row)
    row = {'Type': 'OVERALL', 'N': len(mdf)}
    for tol in tolerances:
        tag = int(tol * 100)
        row[f'Base{tag}'] = mdf[f'baseline_tol{tag}'].mean()
        row[f'Rev{tag}']  = mdf[f'revised_tol{tag}'].mean()
        row[f'Δ{tag}']    = row[f'Rev{tag}'] - row[f'Base{tag}']
    rows.append(row)
    print(pd.DataFrame(rows).set_index('Type').to_string())

# ── dentist effect ────────────────────────────────────────────────────────────

print('\n' + '='*80)
print('DENTIST EFFECT BY TOLERANCE')
print('='*80)

for model in ['Qwen2VL', 'LLaVA']:
    print(f'\n  {model}')
    mdf = df[df['model'] == model]
    for tol in tolerances:
        tag  = int(tol * 100)
        imp  = (mdf[f'revised_tol{tag}'] > mdf[f'baseline_tol{tag}']).sum()
        wor  = (mdf[f'revised_tol{tag}'] < mdf[f'baseline_tol{tag}']).sum()
        unch = (mdf[f'revised_tol{tag}'] == mdf[f'baseline_tol{tag}']).sum()
        n    = len(mdf)
        b    = mdf[f'baseline_tol{tag}'].mean()
        rv   = mdf[f'revised_tol{tag}'].mean()
        print(f'    tol={tag:2d}%  baseline={b:.3f}  revised={rv:.3f}  '
              f'improved={imp:3d} ({imp/n*100:.1f}%)  '
              f'worsened={wor:3d} ({wor/n*100:.1f}%)  '
              f'unchanged={unch:3d} ({unch/n*100:.1f}%)')

# ── numerical-only tolerance ramp ─────────────────────────────────────────────

print('\n' + '='*80)
print('TOLERANCE RAMP — NUMERICAL TYPES ONLY (min/max/mean/median)')
print('='*80)

num_types = ['minimum', 'maximum', 'mean', 'median']
num_df = df[df['type'].isin(num_types)]

for model in ['Qwen2VL', 'LLaVA']:
    print(f'\n  {model}')
    mdf = num_df[num_df['model'] == model]
    rows = []
    for qtype in sorted(mdf['type'].unique()):
        qdf = mdf[mdf['type'] == qtype]
        row = {'Type': qtype}
        for tol in tolerances:
            tag = int(tol * 100)
            row[f'Base{tag}%'] = qdf[f'baseline_tol{tag}'].mean()
            row[f'Rev{tag}%']  = qdf[f'revised_tol{tag}'].mean()
        rows.append(row)
    row = {'Type': 'OVERALL'}
    for tol in tolerances:
        tag = int(tol * 100)
        row[f'Base{tag}%'] = mdf[f'baseline_tol{tag}'].mean()
        row[f'Rev{tag}%']  = mdf[f'revised_tol{tag}'].mean()
    rows.append(row)
    print(pd.DataFrame(rows).set_index('Type').to_string())

Scored 960 results

VERIFICATION — tol20 should match your original script exactly:

--- Qwen2VL ---
  maximum         baseline=0.000  revised=0.000
  mean            baseline=0.113  revised=0.150
  median          baseline=0.000  revised=0.175
  minimum         baseline=0.362  revised=0.300
  nbars           baseline=0.138  revised=0.138
  ngaussians      baseline=0.013  revised=0.263

--- LLaVA ---
  maximum         baseline=0.037  revised=0.000
  mean            baseline=0.000  revised=0.013
  median          baseline=0.000  revised=0.013
  minimum         baseline=0.037  revised=0.013
  nbars           baseline=0.000  revised=0.000
  ngaussians      baseline=0.013  revised=0.150

ACCURACY BY MODEL, QUESTION TYPE, AND TOLERANCE

--- Qwen2VL ---
              N  Base10  Rev10    Δ10  Base20  Rev20    Δ20  Base30  Rev30    Δ30  Base50  Rev50    Δ50
Type                                                                                                   
maximum      80   0.000  0.000  0.